In [1]:
# Load evaluation.json as pandas dataframe and print the first 5 rows
import pandas as pd
import os
from pathlib import Path

ROOT = Path("../..").resolve()
df = pd.read_json(ROOT/'data/processed/evaluation.json')
print(df.head())

                       id                            repo  \
0  synthetic-django_PR_21  kannan-dedsec/synthetic-django   
1  synthetic-django_PR_22  kannan-dedsec/synthetic-django   
2  synthetic-django_PR_23  kannan-dedsec/synthetic-django   
3  synthetic-django_PR_24  kannan-dedsec/synthetic-django   
4  synthetic-django_PR_25  kannan-dedsec/synthetic-django   

                        source_path  \
0                          admin.py   
1                          forms.py   
2  management/commands/seed_data.py   
3                       managers.py   
4                     middleware.py   

                                         source_file  \
0   evaluation_files/synthetic-django_PR_21_admin.py   
1   evaluation_files/synthetic-django_PR_22_forms.py   
2  evaluation_files/synthetic-django_PR_23_manage...   
3  evaluation_files/synthetic-django_PR_24_manage...   
4  evaluation_files/synthetic-django_PR_25_middle...   

                                ground_truth_reviews  
0  [{'

In [2]:
# make a new dataframe with columns id, pr_id (id from old df), line_number, violation_category, review_comment
records = []

for _, row in df.iterrows():
    pr_id = row["id"]
    for review in row["ground_truth_reviews"]:
        records.append({
            "id": len(records),
            "pr_id": pr_id,
            "line_number": review["line_number"],
            "violation_category": review["violation_category"],
            "review_comment": review["review_comment"],
        })

new_df = pd.DataFrame(records, columns=["id", "pr_id", "line_number", "violation_category", "review_comment"])
print(new_df.head())

   id                   pr_id  line_number violation_category  \
0   0  synthetic-django_PR_21            9      unused_import   
1   1  synthetic-django_PR_21           10      unused_import   
2   2  synthetic-django_PR_21           11      unused_import   
3   3  synthetic-django_PR_22           43  naming_convention   
4   4  synthetic-django_PR_22           76  naming_convention   

                        review_comment  
0                    Unused import: os  
1                   Unused import: sys  
2                    Unused import: re  
3  camelCase function name: cleanTitle  
4   camelCase function name: cleanName  


In [3]:
new_df.shape

(675, 5)

In [14]:
new_df['violation_category'].value_counts()

violation_category
unused_import               193
naming_convention           180
indentation                 138
documentation_formatting     92
mutable_default              72
Name: count, dtype: int64

In [5]:
df.shape

(97, 5)

In [6]:
df

,id,repo,source_path,source_file,ground_truth_reviews
0,synthetic-django_PR_21,kannan-dedsec/synthetic-django,admin.py,evaluation_files/synthetic-django_PR_21_admin.py,"[{'line_number': 9, 'violation_category': 'unu..."
1,synthetic-django_PR_22,kannan-dedsec/synthetic-django,forms.py,evaluation_files/synthetic-django_PR_22_forms.py,"[{'line_number': 43, 'violation_category': 'na..."
2,synthetic-django_PR_23,kannan-dedsec/synthetic-django,management/commands/seed_data.py,evaluation_files/synthetic-django_PR_23_manage...,"[{'line_number': 11, 'violation_category': 'un..."
3,synthetic-django_PR_24,kannan-dedsec/synthetic-django,managers.py,evaluation_files/synthetic-django_PR_24_manage...,"[{'line_number': 32, 'violation_category': 'na..."
4,synthetic-django_PR_25,kannan-dedsec/synthetic-django,middleware.py,evaluation_files/synthetic-django_PR_25_middle...,"[{'line_number': 16, 'violation_category': 'un..."
...,...,...,...,...,...
92,synthetic-sklearn_PR_36,kannan-dedsec/synthetic-sklearn,preprocessing.py,evaluation_files/synthetic-sklearn_PR_36_prepr...,"[{'line_number': 15, 'violation_category': 'na..."
93,synthetic-sklearn_PR_37,kannan-dedsec/synthetic-sklearn,regression_models.py,evaluation_files/synthetic-sklearn_PR_37_regre...,"[{'line_number': 16, 'violation_category': 'mu..."
94,synthetic-sklearn_PR_38,kannan-dedsec/synthetic-sklearn,tests/test_models.py,evaluation_files/synthetic-sklearn_PR_38_tests...,"[{'line_number': 1, 'violation_category': 'unu..."
95,synthetic-sklearn_PR_39,kannan-dedsec/synthetic-sklearn,text_classification.py,evaluation_files/synthetic-sklearn_PR_39_text_...,"[{'line_number': 9, 'violation_category': 'unu..."


In [7]:
df['repo'].unique()

<StringArray>
[ 'kannan-dedsec/synthetic-django', 'kannan-dedsec/synthetic-fastapi',
   'kannan-dedsec/synthetic-flask',  'kannan-dedsec/synthetic-pandas',
 'kannan-dedsec/synthetic-sklearn']
Length: 5, dtype: str

In [8]:
new_df.shape

(675, 5)

In [9]:
# Average review comment per PR
avg_comments_per_pr = new_df.groupby('pr_id')['review_comment'].count().mean()
print(f"Average review comments per PR: {avg_comments_per_pr:.2f}")

Average review comments per PR: 7.03


In [10]:
# median, min, max review comments per PR
comments_per_pr = new_df.groupby('pr_id')['review_comment'].count()
median_comments_per_pr = comments_per_pr.median()
min_comments_per_pr = comments_per_pr.min()
max_comments_per_pr = comments_per_pr.max()
print(f"Median review comments per PR: {median_comments_per_pr}")
print(f"Min review comments per PR: {min_comments_per_pr}")
print(f"Max review comments per PR: {max_comments_per_pr}")

Median review comments per PR: 6.0
Min review comments per PR: 1
Max review comments per PR: 33


In [15]:
df['repo'].value_counts()

repo
kannan-dedsec/synthetic-flask      20
kannan-dedsec/synthetic-pandas     20
kannan-dedsec/synthetic-django     19
kannan-dedsec/synthetic-fastapi    19
kannan-dedsec/synthetic-sklearn    19
Name: count, dtype: int64

In [20]:
new_df

,id,pr_id,line_number,violation_category,review_comment
0,0,synthetic-django_PR_21,9,unused_import,Unused import: os
1,1,synthetic-django_PR_21,10,unused_import,Unused import: sys
2,2,synthetic-django_PR_21,11,unused_import,Unused import: re
3,3,synthetic-django_PR_22,43,naming_convention,camelCase function name: cleanTitle
4,4,synthetic-django_PR_22,76,naming_convention,camelCase function name: cleanName
...,...,...,...,...,...
670,670,synthetic-sklearn_PR_40,29,naming_convention,camelCase parameter: experimentName
671,671,synthetic-sklearn_PR_40,32,naming_convention,camelCase parameter: logFile
672,672,synthetic-sklearn_PR_40,55,naming_convention,camelCase function name: formatResults
673,673,synthetic-sklearn_PR_40,32,mutable_default,Mutable default [] for param 'logFile'


In [ ]:
# Repo-wise PR count, review comment count, and comment percentage
repo_stats = (
    new_df.groupby('pr_id', as_index=False)
    .agg(total_comments=('review_comment', 'count'))
    .merge(
        df[['id', 'repo']].rename(columns={'id': 'pr_id'}),
        on='pr_id',
        how='left'
    )
    .groupby('repo', as_index=False)
    .agg(
        total_prs=('pr_id', 'nunique'),
        total_comments=('total_comments', 'sum')
    )
)

repo_stats['comments_percentage'] = (
    repo_stats['total_comments'] / repo_stats['total_comments'].sum() * 100
)

print(pd.DataFrame(repo_stats))

                              repo  total_prs  total_comments  \
0   kannan-dedsec/synthetic-django         19             118   
1  kannan-dedsec/synthetic-fastapi         19             161   
2    kannan-dedsec/synthetic-flask         19             106   
3   kannan-dedsec/synthetic-pandas         20             158   
4  kannan-dedsec/synthetic-sklearn         19             132   

   comments_percentage  
0            17.481481  
1            23.851852  
2            15.703704  
3            23.407407  
4            19.555556  


In [26]:
# Violation category distribution
violation_counts = new_df['violation_category'].value_counts()
print("Violation Category Distribution:")
print(violation_counts)

# percentage distribution of violation categories
violation_percentage = violation_counts / violation_counts.sum() * 100
print("\nViolation Category Percentage Distribution:")
print(violation_percentage)

Violation Category Distribution:
violation_category
unused_import               193
naming_convention           180
indentation                 138
documentation_formatting     92
mutable_default              72
Name: count, dtype: int64

Violation Category Percentage Distribution:
violation_category
unused_import               28.592593
naming_convention           26.666667
indentation                 20.444444
documentation_formatting    13.629630
mutable_default             10.666667
Name: count, dtype: float64
